In [1]:
import os

In [12]:
%pwd

'/Users/User/Documents/mlops/research'

In [8]:
os.chdir("../")

In [9]:
%pwd

'/Users/User/Documents/mlops'

In [28]:
from dataclasses import dataclass
from pathlib import Path

# Whenever you want to create a custom return datatype of a function you can use @dataclass to create one
# In our case we wanted get_data_ingestion_config() fn to return a custom datatype hence we made use of @dataclass

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_train_URL: str
    source_test_URL: str
    train_local_data_file: Path
    test_local_data_file: Path
    unzip_dir: Path

In [29]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories

In [30]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_train_URL=config.source_train_URL,
            source_test_URL=config.source_test_URL,
            train_local_data_file=config.train_local_data_file,
            test_local_data_file=config.test_local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config

In [31]:
import os
import urllib.request as request
import zipfile
from mlProject.utils import logger
from mlProject.utils.common import get_size

In [32]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config


    
    def download_file(self):
        files = [
            (self.config.source_train_URL, self.config.train_local_data_file),
            (self.config.source_test_URL, self.config.test_local_data_file)
        ]
        for url, local_path in files:
            if not os.path.exists(local_path):
                filename, headers = request.urlretrieve(
                    url = url,
                    filename = local_path
                )
                logger.info(f"{filename} download! with following info: \n{headers}")
            else:
                logger.info(f"File already exists of size: {get_size(Path(local_path))}")



    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)

        zip_files = [
            self.config.train_local_data_file,
            self.config.test_local_data_file
        ]

        for file in zip_files:
            with zipfile.ZipFile(file, 'r') as zip_ref:
                zip_ref.extractall(unzip_path)
  

In [34]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-03-05 13:07:45,100: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-03-05 13:07:45,102: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-05 13:07:45,103: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-03-05 13:07:45,104: INFO: common: created directory at: artifacts]
[2026-03-05 13:07:45,104: INFO: common: created directory at: artifacts/data_ingestion]
[2026-03-05 13:07:45,624: INFO: 3549517342: artifacts/data_ingestion/train_data.zip download! with following info: 
Connection: close
Content-Length: 20042
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "36d18c2ccd75aae01f34c1b89a8acc7d6ee8dfdec147d1114a9fd7173e67c566"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: 2346:2AC469:442EDA:4E8A57:69A9C671
Accept-Ranges: bytes
Date: T